# AgentWorkflow & FunctionAgent

The capstone episode builds a tool-calling agent using LangGraph, with LlamaIndex retrieval as "just another tool." This episode builds the **same behavior** — an agent that decides whether to call a retrieval tool — using only LlamaIndex's own native agent system, `FunctionAgent`. No LangChain or LangGraph involved at all.


**Step 1 — Setup.** Configure logging, load API keys, and set the default embedding model (the LLM is set per-agent in Step 3 instead of globally, since this notebook is specifically about agent behavior).


In [1]:
import logging

from dotenv import load_dotenv
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

# Quiet the usual INFO-level noise from httpx and llama_index.
for noisy_logger in ("httpx", "llama_index"):
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)

# Loads keys like OPENAI_API_KEY from .env into os.environ.
load_dotenv()

Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

**Step 2 — Wrap retrieval as a tool.** Build the same VectorStoreIndex as every earlier episode, then wrap a plain Python function that queries it inside a `FunctionTool` — this is what turns "a query engine" into "something an agent can decide to call."


In [2]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.tools import FunctionTool

# The same load -> chunk -> embed -> index -> query engine pipeline as earlier episodes.
documents = SimpleDirectoryReader("data/sample_docs").load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()


def search_anime_docs(query: str) -> str:
    """Search the anime info documents for an answer to the query."""
    # This docstring doubles as the tool description the agent's LLM reads to
    # decide whether this tool is relevant to a given question.
    return str(query_engine.query(query))


# FunctionTool.from_defaults() inspects the function's name, type hints, and
# docstring to build the tool schema the agent's LLM sees — no manual JSON schema needed.
retrieval_tool = FunctionTool.from_defaults(fn=search_anime_docs)

**Step 3 — Build the agent.** `FunctionAgent` is handed the retrieval tool, an LLM, and a system prompt — from here on, deciding _whether_ to call the tool for a given question is the LLM's job, not code you write.


In [3]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI

# gpt-4o-mini instead of the project-wide gpt-4.1-nano default — this LLM has
# to reliably decide WHEN to call the tool, the same reasoning as the capstone.
agent = FunctionAgent(
    tools=[retrieval_tool],
    llm=OpenAI(model="gpt-4o-mini"),
    system_prompt="You are a helpful assistant with access to an anime info search tool.",
)

**Step 4 — Watch it decide.** Two questions clearly need the anime tool, one clearly doesn't — the agent should call the tool for the first two and just answer directly for the third, proving the tool-use decision is genuinely conditional, not hardcoded.


In [4]:
from llama_index.core.agent.workflow.workflow_events import AgentWorkflowStartEvent
from llama_index.core.workflow import Context

questions = [
    "What is Naruto's signature technique?",
    "How many Dragon Balls are needed to summon Shenron?",
    "What's the capital of France?",  # control case — should NOT trigger the tool
]

for question in questions:
    # agent.run() is async: it may call the tool internally. FunctionAgent.run()
    # has a deprecated overload keyed on `user_msg`/`start_event` as keywords —
    # passing a fresh Context positionally as the first arg is what actually
    # routes to the non-deprecated Workflow.run(ctx, start_event=...) overload.
    ctx = Context(agent)
    response = await agent.run(ctx, start_event=AgentWorkflowStartEvent(user_msg=question))
    print(f"Q: {question}\nA: {response}\n")

Q: What is Naruto's signature technique?
A: Naruto's signature technique is the Rasengan, which is a swirling ball of concentrated chakra. It was taught to him by his mentor, Jiraiya.

Q: How many Dragon Balls are needed to summon Shenron?
A: To summon Shenron, seven Dragon Balls are needed.

Q: What's the capital of France?
A: The capital of France is Paris.



### Summary

- `FunctionAgent` is LlamaIndex's own tool-calling agent, built on its `Workflow` engine — `agent.run()` returns an awaitable, which is why every call above uses `await` directly in the notebook cell.
- Behavior-wise, this is functionally the same agent as the capstone's LangGraph version: an LLM decides whether a question needs the retrieval tool, calls it when it does, and answers directly when it doesn't (see the control question above).
- Reach for `FunctionAgent`/`AgentWorkflow` when your orchestration needs are simple and LlamaIndex-native; reach for LangGraph (as the capstone does) when you need more complex graph-shaped control flow, especially across tools and frameworks beyond LlamaIndex.
